# Image Classification Transfer learning with YOLOv11



In [1]:
!pip install ultralytics

  Using cached ultralytics-8.4.43-py3-none-any.whl.metadata (39 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached polars-1.40.1-py3-none-any.whl.metadata (10 kB)
  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
  Using cached polars_runtime_32-1.40.1-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached ultralytics-8.4.43-py3-none-any.whl (1.2 MB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
Using cached polars-1.40.1-py3-none-any.whl (828 kB)
Using cached polars_runtime_32-1.40.1-cp310-abi3-win_amd64.whl (51.8 MB)
Using cached ultralytics_thop-2.0.19-py3-none-any.whl (28 kB)

   ---------------------------------------- 0/5 [polars-runtime-32]
   ---------------------------------------- 0/5 [polars-runtime-32]
   ---------------------------------------- 0/5 [polars-runtime-32]
   ---------------------------------------- 0/5 [polars-runtime-32]
   ----------------------------------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import torch
import torchvision.transforms as T

from ultralytics import YOLO
from ultralytics.data.dataset import ClassificationDataset
from ultralytics.models.yolo.classify import (
    ClassificationTrainer,
    ClassificationValidator,
)  # Import ClassificationValidator


class CustomizedDataset(ClassificationDataset):
    """A customized dataset class for image classification with enhanced data augmentation transforms."""

    def __init__(self, root: str, args, augment: bool = False, prefix: str = ""):
        """Initialize a customized classification dataset with enhanced data augmentation transforms."""
        super().__init__(root, args, augment, prefix)
        train_transforms = T.Compose(
            [
                T.Resize((args.imgsz, args.imgsz)),
                T.RandomHorizontalFlip(p=args.fliplr),
                T.RandomVerticalFlip(p=args.flipud),
                T.RandAugment(interpolation=T.InterpolationMode.BILINEAR),
                T.ColorJitter(
                    brightness=args.hsv_v,
                    contrast=args.hsv_v,
                    saturation=args.hsv_s,
                    hue=args.hsv_h,
                ),
                T.ToTensor(),
                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
                T.RandomErasing(p=args.erasing, inplace=True),
            ]
        )
        val_transforms = T.Compose(
            [
                T.Resize((args.imgsz, args.imgsz)),
                T.ToTensor(),
                T.Normalize(mean=torch.tensor(0), std=torch.tensor(1)),
            ]
        )
        self.torch_transforms = train_transforms if augment else val_transforms


class CustomizedTrainer(ClassificationTrainer):
    """A customized trainer class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str, mode: str = "train", batch=None):
        """Build a customized dataset for classification training and the validation during training."""
        return CustomizedDataset(root=img_path, args=self.args, augment=mode == "train", prefix=mode)


class CustomizedValidator(ClassificationValidator):  # Now ClassificationValidator is defined
    """A customized validator class for YOLO classification models with enhanced dataset handling."""

    def build_dataset(self, img_path: str, mode: str = "train"):
        """Build a customized dataset for classification standalone validation."""
        return CustomizedDataset(root=img_path, args=self.args, augment=mode == "train", prefix=self.args.split)

### Download Dataset

In [4]:
!pip install gdown
import gdown

# Google Drive file ID
file_id = "1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl"
# Download destination filename
output = "myfile.zip"

# Download the file
gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

  Using cached gdown-6.0.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached PySocks-1.7.1-py3-none-any.whl.metadata (13 kB)
Using cached gdown-6.0.0-py3-none-any.whl (18 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
Using cached PySocks-1.7.1-py3-none-any.whl (16 kB)

   ---------------------------------------- 0/4 [soupsieve]
   -------------------- ------------------- 2/4 [beautifulsoup4]
   -------------------- ------------------- 2/4 [beautifulsoup4]
   -------------------- ------------------- 2/4 [beautifulsoup4]
   ------------------------------ --------- 3/4 [gdown]
   ------------------------------ --------- 3/4 [gdown]
   ---------------------------------------- 4/4 [gdown]




[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Downloading...
From (original): https://drive.google.com/uc?id=1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl
From (redirected): https://drive.google.com/uc?id=1TCU1nqgIe1R_dW6LTkRxlufHlCCazyJl&confirm=t&uuid=b6e06d7e-076f-41b5-975a-a9844cc8a800
To: c:\Users\Jagdish singh\DEEP-LEARNING\Transfer_Learning_img_classfxn\myfile.zip
100%|██████████| 63.9M/63.9M [00:18<00:00, 3.47MB/s]


'myfile.zip'

In [5]:
import zipfile

with zipfile.ZipFile("myfile.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

### Custom Training or Transfer learning

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n-cls.pt")
#model.train(data="imagenet1000", trainer=CustomizedTrainer, epochs=10, imgsz=224, batch=64)
model.train(data="/content/dataset/train", trainer=CustomizedTrainer, epochs=1, imgsz=224, batch=64)

In [ ]:
from ultralytics import YOLO

# Point to the local file you just moved
model_path = r"C:\Users\Jagdish singh\DEEP-LEARNING\Transfer_Learning_img_classfxn\yolo11n-cls.pt"
model = YOLO(model_path)

# Now run the prediction
results = model(r"C:\Users\Jagdish singh\DEEP-LEARNING\Transfer_Learning_img_classfxn\vine_snake.jpg")

In [ ]:
# CLI Start training from a pretrained *.pt model
#!yolo classify train data=imagenet10 model=yolo11n-cls.pt epochs=5 imgsz=224

In [ ]:
metrics = model.val(data="/content/dataset/valid", validator=CustomizedValidator, imgsz=224, batch=64)

Ultralytics 8.3.159 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
YOLO11n-cls summary (fused): 47 layers, 1,528,586 parameters, 0 gradients, 3.2 GFLOPs
WARNING ⚠️ Dataset 'split=train' not found at /content/dataset/valid/train
Found 364 images in subdirectories. Attempting to split...
Splitting /content/dataset/valid (2 classes, 364 images) into 80% train, 20% val...
Split complete in /content/dataset/valid_split ✅
train: /content/dataset/valid_split/train... found 290 images in 2 classes ✅ 
val: /content/dataset/valid_split/val... found 74 images in 2 classes ✅ 
test: None...
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 720.4±250.4 MB/s, size: 25.3 KB)


val: Scanning /content/dataset/valid_split/val... 74 images, 0 corrupt: 100%|██████████| 74/74 [00:00<00:00, 5338.65it/s]

val: New cache created: /content/dataset/valid_split/val.cache



               classes   top1_acc   top5_acc: 100%|██████████| 2/2 [00:01<00:00,  1.18it/s]


                   all      0.905          1
Speed: 0.6ms preprocess, 4.8ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to runs/classify/train2


In [ ]:
metrics.top1  # top1 accuracy


0.9054054021835327

In [ ]:
metrics.top5  # top5 accuracy

1.0

In [ ]:
model = YOLO("/content/runs/classify/train/weights/best.pt")  # load a custom model

# Predict with the model
results = model("/content/dataset/test/daisy/14333681205_a07c9f1752_m_jpg.rf.6ff96d9fe33f0bd19a18425f32d470b1.jpg")


image 1/1 /content/dataset/test/daisy/14333681205_a07c9f1752_m_jpg.rf.6ff96d9fe33f0bd19a18425f32d470b1.jpg: 224x224 daisy 1.00, dandelion 0.00, 6.5ms
Speed: 4.5ms preprocess, 6.5ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)


In [ ]:
# Predict with the model
results = model("/content/dataset/test/dandelion/16159487_3a6615a565_n_jpg.rf.6d473a1fe680a3e930f3ff28464c46a9.jpg")


image 1/1 /content/dataset/test/dandelion/16159487_3a6615a565_n_jpg.rf.6d473a1fe680a3e930f3ff28464c46a9.jpg: 224x224 dandelion 1.00, daisy 0.00, 6.2ms
Speed: 5.7ms preprocess, 6.2ms inference, 0.1ms postprocess per image at shape (1, 3, 224, 224)
